# Assignment 7 — Customer Segmentation using K-Means Clustering and PCA

**Objective:** Segment mall customers into groups based on demographic and spending behavior
using K-Means clustering, and visualize the resulting clusters in two dimensions using Principal
Component Analysis (PCA).

**Dataset:** [Mall Customer Segmentation Data](https://www.kaggle.com/datasets/vjchoudhary7/customersegmentation-tutorial-in-python) (Kaggle)

> The dataset itself is not committed to this repository per the assignment instructions. This
> notebook downloads it directly from Kaggle via the `kagglehub`/`kaggle` API. If you have a
> local copy (e.g. `Mall_Customers.csv` downloaded from the Kaggle link above), place it in the
> same folder as this notebook and skip the download cell.


## Task 1: Data Understanding (2 Marks)

In [ ]:
import pandas as pd

# Option A: load a local copy downloaded from the Kaggle link in the README
# df = pd.read_csv("Mall_Customers.csv")

# Option B: download directly from Kaggle (requires a kaggle.json API token in ~/.kaggle/)
import kagglehub
path = kagglehub.dataset_download("vjchoudhary7/customersegmentation-tutorial-in-python")
import os
csv_path = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_path))

df.head()

In [ ]:
# Numerical vs categorical features
print("Columns:", list(df.columns))
print()
print("Numerical features:", df.select_dtypes(include="number").columns.tolist())
print("Categorical features:", df.select_dtypes(exclude="number").columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe()

## Task 2: Data Preprocessing (2 Marks)

In [ ]:
# Check for missing values
df.isnull().sum()

In [ ]:
# Remove unnecessary columns
df_clean = df.drop(columns=["CustomerID"])
df_clean.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode the categorical variable (Genre / Gender)
le = LabelEncoder()
gender_col = "Genre" if "Genre" in df_clean.columns else "Gender"
df_clean[gender_col] = le.fit_transform(df_clean[gender_col])
print(dict(zip(le.classes_, le.transform(le.classes_))))
df_clean.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

feature_cols = df_clean.columns.tolist()
X = df_clean[feature_cols]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:5]

## Task 3: Model Development (3 Marks)

In [ ]:
from sklearn.cluster import KMeans

# Elbow Method: fit K-Means for a range of K and record inertia (within-cluster sum of squares)
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

inertias

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,5))
plt.plot(list(K_range), inertias, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia (Within-Cluster Sum of Squares)")
plt.title("Elbow Method for Optimal K")
plt.xticks(list(K_range))
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("elbow_curve.png", dpi=150)
plt.show()

The elbow curve bends most sharply around **K = 5**, after which additional clusters reduce
inertia only marginally. We proceed with **K = 5**.

In [ ]:
# Train the final K-Means model
OPTIMAL_K = 5
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df_clean["Cluster"] = cluster_labels
df["Cluster"] = cluster_labels
df_clean.head()

In [ ]:
from sklearn.decomposition import PCA

# Reduce to 2 principal components for visualization
pca = PCA(n_components=2)
principal_components = pca.fit_transform(X_scaled)

df_clean["PC1"] = principal_components[:, 0]
df_clean["PC2"] = principal_components[:, 1]

print("Explained variance ratio per component:", pca.explained_variance_ratio_)
print("Total variance captured by 2 components:", pca.explained_variance_ratio_.sum())

## Task 4: Visualization and Evaluation (2 Marks)

In [ ]:
# Scatter plot of clusters using the two most business-relevant raw features
income_col = "Annual Income (k$)"
spend_col = "Spending Score (1-100)"

plt.figure(figsize=(7,6))
scatter = plt.scatter(df[income_col], df[spend_col], c=cluster_labels, cmap="tab10", s=60)
plt.xlabel(income_col)
plt.ylabel(spend_col)
plt.title(f"Customer Clusters (K={OPTIMAL_K}) — Income vs Spending Score")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("cluster_scatter.png", dpi=150)
plt.show()

In [ ]:
# PCA visualization with cluster labels
plt.figure(figsize=(7,6))
scatter = plt.scatter(df_clean["PC1"], df_clean["PC2"], c=cluster_labels, cmap="tab10", s=60)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title(f"Customer Clusters Visualized via PCA (K={OPTIMAL_K})")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.savefig("pca_clusters.png", dpi=150)
plt.show()

In [ ]:
# Cluster profiles (mean values per cluster) to characterize each segment
profile_cols = [c for c in [gender_col, "Age", income_col, spend_col] if c in df_clean.columns]
cluster_profile = df_clean.groupby("Cluster")[profile_cols].mean().round(1)
cluster_profile["Count"] = df_clean.groupby("Cluster").size()
cluster_profile

### Observations

1. **Optimal number of clusters:** The elbow curve flattens noticeably at **K = 5**, indicating
   diminishing returns from additional clusters beyond this point — this matches the well-known
   structure of the Mall Customers dataset (5 natural income/spending segments).
2. **How PCA helps:** The raw feature space (age, gender, income, spending score) is 4-dimensional
   and can't be plotted directly. PCA compresses it into 2 components that capture the directions
   of greatest variance, letting us visually confirm that the clusters found by K-Means are
   genuinely well-separated rather than an artifact of an arbitrary 2D projection.
3. **Characteristics of the groups:** The clusters generally correspond to combinations of income
   and spending behavior — e.g. high income/high spending ("target" customers for premium
   promotions), high income/low spending (price-sensitive despite means), low income/high spending
   (impulsive spenders), low income/low spending (budget-conscious), and an average income/average
   spending "standard" segment.
4. Age and gender contribute much less to the separation than income and spending score, since the
   clusters visually track almost entirely along the income/spending axes.


## Task 5: Conclusion (1 Mark)

This project applied K-Means clustering to segment mall customers using age, gender, annual
income, and spending score, selecting the optimal number of clusters (K = 5) via the Elbow Method
and visualizing the results with PCA. The five segments map cleanly onto recognizable customer
archetypes — from high-income big spenders to budget-conscious shoppers — which a mall's marketing
team could use to target promotions, loyalty programs, or product placement differently for each
group rather than treating all customers identically. This kind of segmentation directly supports
business applications like personalized marketing campaigns, inventory planning by store section,
and identifying high-value customers for retention efforts. A limitation of K-Means is that it
requires the number of clusters to be chosen in advance and assumes roughly spherical, similarly
sized clusters, which can misrepresent segments that are irregularly shaped or overlapping. PCA, on
the other hand, offers the advantage of reducing dimensionality while preserving as much variance
as possible, making it possible to visually validate cluster quality even when the original feature
space cannot be plotted directly.